# 단숨 FunctionGemma 270M 학습
한국어 숫자·통화·생활 단위 변환과 추가 질문을 위한 FunctionGemma 함수 호출 모델을 학습하고 `janyty/browsertools-functiongemma-270m`에 공개합니다.

실행 전 Colab의 열쇠 아이콘에서 `HF_TOKEN`을 등록하고 노트북 액세스를 허용하세요. 토큰을 코드 셀이나 채팅에 직접 입력하지 않습니다.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), '런타임 > 런타임 유형 변경에서 T4 GPU 또는 L4 GPU를 선택하세요.'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
from google.colab import files
uploaded = files.upload()
assert 'functiongemma_training_bundle.zip' in uploaded, 'functiongemma_training_bundle.zip을 선택하세요.'
!rm -rf /content/browsertool_korean
!mkdir -p /content/browsertool_korean
!unzip -q -o functiongemma_training_bundle.zip -d /content/browsertool_korean
%cd /content/browsertool_korean

In [ ]:
!pip -q install -r requirements-training.txt

In [ ]:
from google.colab import userdata
from huggingface_hub import login, HfApi
token = userdata.get('HF_TOKEN')
assert token, 'Colab 비밀 설정에 HF_TOKEN을 추가하고 이 노트북의 액세스를 허용하세요.'
login(token=token, add_to_git_credential=False)
api = HfApi(token=token)
api.create_repo('janyty/browsertools-functiongemma-270m', repo_type='model', private=False, exist_ok=True)
print('Hugging Face 로그인과 공개 저장소 준비 완료')

In [ ]:
!python scripts/build_functiongemma_dataset.py
!python scripts/train_functiongemma.py --epochs 5 --output /content/dansum-functiongemma-270m --hub-repo janyty/browsertools-functiongemma-270m

In [ ]:
import json
from pathlib import Path
result = json.loads(Path('/content/dansum-functiongemma-270m/functiongemma_evaluation.json').read_text())
print('독립 테스트 정확도:', f"{result['tool_call_accuracy']:.2%}")
print(json.dumps(result['per_tool'], ensure_ascii=False, indent=2))